In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from rich import print
import scipy


In [ ]:
import sys
sys.path.append('/Volumes/B/Projects/2025Project/SpatialAnalysis/Code')
import SpatialPyALab as lab  # type: ignore

In [ ]:
data_dir = '/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC_spatial_V2/Data/SpatialData/0_Samples/'
sample_id = os.path.basename(os.getcwd())
root_dir = data_dir+sample_id
root_dir

In [ ]:
h5ad_file = os.path.join(root_dir,'preprocess/mask_tissue.h5ad')
adata_spot = sc.read_h5ad(h5ad_file)

In [ ]:
coords_key='spatial_manual_fine'
mu_scale = adata_spot.uns['mu_scale'][coords_key]
hex_edge = 15
mesh_adata,orig_idx,new_idx = lab.mesh.create_meshed_adata(adata_spot,
                                                           coords_key=coords_key,
                                                           px_by_um = mu_scale, 
                                                           spot_diameter_um = hex_edge)
print(f"Cells: [bold green]{mesh_adata.shape[0]}[/bold green], Genes: [bold green]{mesh_adata.shape[1]}[/bold green]")

In [ ]:
GLMPCA_adata = mesh_adata.copy()

In [ ]:
lab.tl.deviance(GLMPCA_adata, fam = 'binomial')
lab.pl.plot_deviance(GLMPCA_adata)

In [ ]:
from scipy.stats import chi2
alpha = 0.001  
n_df = 500  
deviance = GLMPCA_adata.var['deviance'].values
sorted_idx = np.argsort(deviance)[::-1]
sorted_deviance = deviance[sorted_idx]
critical_deviance = chi2.isf(alpha, n_df)
max_genes = (sorted_deviance > critical_deviance).sum()

In [ ]:
n_genes = 10000
n_genes = n_genes if max_genes < n_genes else max_genes
print(f'The lowest deviance of top {n_genes} genes is {sorted_deviance[n_genes-1]}')

In [ ]:
glmpca_res = lab.tl.glmpca(GLMPCA_adata, fam = 'poi', npcs = 50, penalty = 10,top_n = n_genes,  
                           verbose = True, n_retry = 5)

In [ ]:
GLMPCA_adata.obsm['X_glmpca'] = glmpca_res['factors']
GLMPCA_adata.uns['glmpca'] = {'loadings': glmpca_res['loadings'],
                            'coefX': glmpca_res['coefX'],
                            'coefZ': glmpca_res['coefZ'],
                            'dev': glmpca_res['dev'],
                            'family': glmpca_res['glmpca_family'].glmpca_fam,
                           }

In [ ]:
GLMPCA_adata.layers['raw'] = GLMPCA_adata.X.copy()
sc.pp.normalize_total(GLMPCA_adata, target_sum=1e6, inplace=True)
GLMPCA_adata.layers['norm'] = GLMPCA_adata.X.copy()
sc.pp.log1p(GLMPCA_adata)
GLMPCA_adata.layers['log1p'] = GLMPCA_adata.X.copy()

In [ ]:
h5ad_file = os.path.join(root_dir,'GLMPCA/GLMPCA_adata.h5ad')
GLMPCA_adata = sc.read_h5ad(h5ad_file)

In [ ]:
sc.pp.neighbors(GLMPCA_adata, use_rep='X_glmpca')
sc.tl.umap(GLMPCA_adata)
sc.tl.leiden(GLMPCA_adata, resolution=1, key_added='leiden')

In [ ]:
n_clusters = GLMPCA_adata.obs['leiden'].nunique()
GLMPCA_colors = {
    f'{i}': lab.pl.GLM_PCA_Hex_List[f'leiden_{i}']
    for i in range(n_clusters)
}

In [ ]:
sc.pl.umap(GLMPCA_adata, color = 'leiden',palette=GLMPCA_colors)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (8, 8))
sc.pl.spatial(GLMPCA_adata, 
              color = 'leiden',
              spot_size = 90,
              palette=GLMPCA_colors, 
              img_key = None, 
              ax=axs, 
              show = False)

In [ ]:
if 'deviance' in GLMPCA_adata.uns:
    print(GLMPCA_adata.uns['deviance'])  
    del GLMPCA_adata.uns['deviance']

In [ ]:
os.makedirs(os.path.join(root_dir,'GLMPCA'), exist_ok=True)
GLMPCA_adata.write(filename=os.path.join(root_dir,'GLMPCA/GLMPCA_adata.h5ad'))